# GRADE RAG Chatbot (Simplified)

Clean RAG pipeline over the GRADE Book — no LangGraph, no agents, no Docker.

**Architecture:**
```
scrape_and_chunk(urls) → build_index() → rag_answer(query)
```

**Setup:** copy `.env.example` to `.env` and fill in:
```
OPENROUTER_API_KEY=...
MODEL_NAME=google/gemini-2.5-flash
```

In [15]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from dotenv import load_dotenv
load_dotenv()

from rag_core import get_pdf_paths, load_or_build_index, rag_answer, build_llm

## 1. Configuration

In [16]:
GRADE_URLS = [
    "https://book.gradepro.org/guideline/overview-of-the-grade-approach",
    "https://book.gradepro.org/guideline/the-development-methods-of-grade",
    "https://book.gradepro.org/guideline/requirements-for-claiming-the-use-of-grade",
    "https://book.gradepro.org/guideline/questions-about-interventions-diagnostic-test-prognosis-and-exposures",
    "https://book.gradepro.org/guideline/risk-of-bias-randomized-trials",
    "https://book.gradepro.org/guideline/inconsistency",
    "https://book.gradepro.org/guideline/imprecision",
    "https://book.gradepro.org/guideline/dissemination-bias",
]

GRADE_SYSTEM_PROMPT = (
    "You are an expert assistant specialising in the GRADE methodology for evidence-based "
    "healthcare guidelines. Answer the user's question based ONLY on the retrieved GRADE Book "
    "content below. Do not fabricate or speculate beyond what is in the context. "
    "If the context does not contain enough information, say so explicitly. "
    "Use precise GRADE terminology (certainty of evidence, rating down/up, domains, etc.)."
)

GRADE_PDF_DIR = "data/grade_pdfs"
PARSED_GRADE_PDF_DIR = "data/parsed_grade"
CACHE_DIR = ".faiss_cache/grade"

grade_pdf_paths = get_pdf_paths(GRADE_PDF_DIR)
print(f"Found {len(grade_pdf_paths)} GRADE PDFs")

Found 43 GRADE PDFs


## 2. Build / Load Index

First run scrapes all 8 GRADE Book pages (~40 s) and saves a local FAISS cache.  
Subsequent runs load from cache instantly.

In [17]:
index = load_or_build_index(
    urls=GRADE_URLS,
    pdf_paths=grade_pdf_paths,
    cache_path=CACHE_DIR,
    parsed_pdf_cache_dir=PARSED_GRADE_PDF_DIR,
)
llm = build_llm()
print("Ready.")

Building index from scratch...
  Scraping: https://book.gradepro.org/guideline/overview-of-the-grade-approach
    -> 40 chunks
  Scraping: https://book.gradepro.org/guideline/the-development-methods-of-grade
    -> 10 chunks
  Scraping: https://book.gradepro.org/guideline/requirements-for-claiming-the-use-of-grade
    -> 14 chunks
  Scraping: https://book.gradepro.org/guideline/questions-about-interventions-diagnostic-test-prognosis-and-exposures
    -> 15 chunks
  Scraping: https://book.gradepro.org/guideline/risk-of-bias-randomized-trials
    -> 38 chunks
  Scraping: https://book.gradepro.org/guideline/inconsistency
    -> 21 chunks
  Scraping: https://book.gradepro.org/guideline/imprecision
    -> 25 chunks
  Scraping: https://book.gradepro.org/guideline/dissemination-bias
    -> 22 chunks
Total documents: 185
  Loading parsed PDF from cache: 1.pdf
    -> 19 documents
  Loading parsed PDF from cache: 10.pdf
    -> 18 documents
  Loading parsed PDF from cache: 11.pdf
    -> 7 documen

## 3. Ask a Question

In [18]:
query = "What are the four levels of certainty of evidence in GRADE?"
answer, contexts = rag_answer(query, index, llm, k=5, system_prompt=GRADE_SYSTEM_PROMPT)
print("Answer:\n", answer)
print("\n--- Retrieved contexts ---")
for i, ctx in enumerate(contexts, 1):
    print(f"[{i}] {ctx[:200]}...")

Answer:
 The four levels of certainty of evidence in GRADE are:

*   High
*   Moderate
*   Low
*   Very low

--- Retrieved contexts ---
[1] GRADE specifies four categories for the quality of a body of evidence
Although the quality of evidence represents a continuum, the GRADE approach results in an assessment of the quality of a body of e...
[2]  a particular range."
(5,6)
And for evidence from reviews of qualitative research: “an assessment of the extent to which the review finding is a reasonable representation of the phenomenon of interest...
[3] Certainty of evidence
What is the overall certainty of the evidence of effects?
Answer based on the certainty assessment in the GRADE evidence profile....
[4] users to make judgments within these domains. Signalling questions, e.g. about the priority of a problem, are used to answer questions for each criterion.
Requirements for basic GRADE usage
The GRADE ...
[5] ertainty of the evidence therefore begins with a rating of high certainty. Th

## 4. Convenience Functions (for evaluation notebooks)

In [19]:
def ask_grade(question: str) -> str:
    answer, _ = rag_answer(question, index, llm, system_prompt=GRADE_SYSTEM_PROMPT)
    return answer

def get_grade_contexts(question: str, k: int = 5) -> list[str]:
    _, contexts = rag_answer(question, index, llm, k=k, system_prompt=GRADE_SYSTEM_PROMPT)
    return contexts

In [22]:
questions = [
      "What is a potential risk of combining tests with therapeutic interventions?",   # Basic
      "What can a multiple intervention comparison framework help with?",                        # Intermediate
      "What challenge may arise when comparing different diagnostic tests?",  #
      "What are the causes of incoherence?"

  ]
for q in questions:
      ans, ctx = rag_answer(q, index, llm, k=5, system_prompt=GRADE_SYSTEM_PROMPT)
      print(f"Q: {q}\nA: {ans}\n")

Q: What is a potential risk of combining tests with therapeutic interventions?
A: The GRADE book content does not directly address the potential risks of combining tests with therapeutic interventions. However, it highlights that "testing, including for diagnosis, screening, and monitoring, demands interventions that become part of the overall strategy to management and that have consequences that require consideration."

The text also notes that for tests, "accuracy studies typically provide low quality evidence for making recommendations due to indirectness of the outcomes, similar to surrogate outcomes for treatments." This suggests that relying solely on test accuracy without considering the downstream consequences of the therapeutic interventions that follow could be a risk.

Furthermore, the example of cervical screening guidelines illustrates that women with a positive test (including false positives) would undergo further management with therapies, and "both groups would experi

## 5. Optional Gradio UI

In [21]:
import gradio as gr

def gradio_fn(question, history):
    answer, _ = rag_answer(question, index, llm, system_prompt=GRADE_SYSTEM_PROMPT)
    return answer

demo = gr.ChatInterface(
    fn=gradio_fn,
    title="GRADE Book Assistant",
    description="Ask questions about the GRADE methodology. Answers are grounded in the GRADE Book.",
)
# Uncomment to launch:
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
